In [5]:
!pip install pandas numpy scikit-learn matplotlib seaborn nltk vaderSentiment


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.4 MB/s eta 0:00:00


In [6]:
import pandas as pd
df = pd.read_csv("/content/IMDB Dataset.csv")

df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r"<.*?>", "", text)  # remove HTML tags
    text = re.sub(r"[^a-z\s]", "", text)  # remove punctuation/numbers
    text = " ".join([word for word in text.split() if word not in stop_words])  # remove stopwords
    return text

df['clean_review'] = df['review'].apply(clean_text)
df.head()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching oz episode yo...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


In [4]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

df['clean_review'] = df['clean_review'].apply(lemmatize_text)


[nltk_data] Downloading package wordnet to /root/nltk_data...


In [8]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

def get_neutral(text):
    score = analyzer.polarity_scores(text)['compound']
    if -0.05 <= score <= 0.05:
        return 'Neutral'
    return None

df['emotion'] = df['clean_review'].apply(get_neutral)


In [9]:
contemplation_keywords = ["think","wonder","ponder","reflect","consider"]
concern_keywords = ["worried","anxious","fear","trouble","problem"]

def assign_emotion(text):
    if any(word in text for word in contemplation_keywords):
        return "Contemplation"
    elif any(word in text for word in concern_keywords):
        return "Concern"
    return None

df.loc[df['emotion'].isna(), 'emotion'] = df.loc[df['emotion'].isna(), 'clean_review'].apply(assign_emotion)


In [10]:
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('omw-1.4')  # for extended synonyms


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [11]:
def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().lower().replace("_", " "))
    return synonyms


In [12]:
contemplation_keywords = [
    "think", "thinking", "wonder", "ponder", "reflect", "consider",
    "question", "curious", "musing", "speculate", "deliberate",
    "imagine", "observe", "contemplate", "analyze", "evaluate",
    "suppose", "guess", "reason", "study"
]

In [13]:
concern_keywords = [
    "worried", "worry", "anxious", "anxiety", "fear", "trouble",
    "problem", "concerned", "nervous", "uneasy", "panic", "stress",
    "uneasiness", "doubt", "scared", "disturb", "tense", "apprehensive",
    "afraid"
]

In [14]:
# Function to expand a list of keywords with synonyms
def expand_keywords(keywords):
    expanded = set(keywords)  # start with original words
    for word in keywords:
        expanded.update(get_synonyms(word))
    return list(expanded)

# Expand both lists
contemplation_keywords = expand_keywords(contemplation_keywords)
concern_keywords = expand_keywords(concern_keywords)


In [15]:
def assign_emotion(text):
    if any(word in text for word in contemplation_keywords):
        return "Contemplation"
    elif any(word in text for word in concern_keywords):
        return "Concern"
    return None

# Apply the labeling function only to reviews that are not already labeled
df.loc[df['emotion'].isna(), 'emotion'] = df.loc[df['emotion'].isna(), 'clean_review'].apply(assign_emotion)


In [16]:
def joy_sadness(row):
    if pd.isna(row['emotion']):  # if still unlabeled
        return "Joy" if row['sentiment'] == "positive" else "Sadness"
    return row['emotion']

df['emotion'] = df.apply(joy_sadness, axis=1)


In [17]:
df['emotion'].value_counts()


,count
emotion,
Contemplation,45150
Concern,3692
Neutral,547
Joy,317
Sadness,294


In [18]:
from sklearn.model_selection import train_test_split

X = df['clean_review']  # features (text)
y = df['emotion']       # labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [20]:
# Balance the training data using oversampling
from imblearn.over_sampling import RandomOverSampler
import numpy as np

ros = RandomOverSampler(random_state=42)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train_vec, y_train)

print("Before:", dict(zip(*np.unique(y_train, return_counts=True))))
print("After:", dict(zip(*np.unique(y_train_balanced, return_counts=True))))


Before: {'Concern': np.int64(2953), 'Contemplation': np.int64(36120), 'Joy': np.int64(254), 'Neutral': np.int64(438), 'Sadness': np.int64(235)}
After: {'Concern': np.int64(36120), 'Contemplation': np.int64(36120), 'Joy': np.int64(36120), 'Neutral': np.int64(36120), 'Sadness': np.int64(36120)}


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight=None,  # turn off since we balanced already
    random_state=42,
    n_jobs=-1
)

# Train on balanced data
rf_model.fit(X_train_balanced, y_train_balanced)

# Predict on original test set
y_pred_rf = rf_model.predict(X_test_vec)

print("Classification Report:\n")
print(classification_report(y_test, y_pred_rf))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_rf))


Classification Report:

               precision    recall  f1-score   support

      Concern       0.95      0.11      0.20       739
Contemplation       0.91      1.00      0.95      9030
          Joy       0.80      0.06      0.12        63
      Neutral       1.00      0.01      0.02       109
      Sadness       1.00      0.05      0.10        59

     accuracy                           0.91     10000
    macro avg       0.93      0.25      0.28     10000
 weighted avg       0.92      0.91      0.88     10000

Confusion Matrix:

[[  81  658    0    0    0]
 [   2 9027    1    0    0]
 [   0   59    4    0    0]
 [   2  106    0    1    0]
 [   0   56    0    0    3]]


In [21]:
while True:
    sample_review = input("Enter a movie review (or type 'quit' to exit): ")

    if sample_review.lower() == "quit":
        print("Exiting live demo...")
        break

    # Preprocess + Vectorize
    sample_clean = preprocess_text(sample_review)
    sample_vec = tfidf.transform([sample_clean])

    # Prediction
    prediction = model.predict(sample_vec)[0]
    print("Custom Review Prediction →", prediction)
    print("-" * 50)